# Transformers

## Learning Objectives
1. Implement a complete transformer block in NumPy: scaled dot-product attention (QKV) + FFN + residual + LayerNorm.
2. Build a multi-layer transformer encoder with PyTorch's `TransformerEncoderLayer` with OOM handling.
3. Fine-tune a HuggingFace AutoModel for sequence classification as a production NLP pattern.
4. Compare Transformer vs LSTM on long sequences to understand parallelism vs memory trade-offs.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: Transformer Block from Scratch — NumPy (QKV + FFN + Residual + LN)

In [ ]:
# Full transformer block: Multi-Head Self-Attention -> Add&Norm -> FFN -> Add&Norm
# Scaled dot-product attention: Attention(Q,K,V) = softmax(QK^T/sqrt(d_k)) V

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def layer_norm(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Layer normalization across last dimension."""
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True) + eps
    return (x - mean) / std


def scaled_dot_product_attention(
    Q: np.ndarray, K: np.ndarray, V: np.ndarray
) -> tuple:
    """Compute attention output and weights.
    
    Q, K, V: (seq_len, d_k)
    Returns: (output, weights) where output is (seq_len, d_k).
    """
    d_k = Q.shape[-1]
    # Scale by sqrt(d_k) to prevent softmax saturation
    scores = Q @ K.T / np.sqrt(d_k)  # (seq_len, seq_len)
    weights = softmax(scores)         # (seq_len, seq_len)
    output = weights @ V              # (seq_len, d_k)
    return output, weights


def transformer_block(
    X: np.ndarray,
    W_q: np.ndarray, W_k: np.ndarray, W_v: np.ndarray, W_o: np.ndarray,
    W_1: np.ndarray, b_1: np.ndarray,
    W_2: np.ndarray, b_2: np.ndarray,
) -> tuple:
    """One transformer encoder block.
    
    X: (seq_len, d_model)
    Returns (output, attention_weights).
    """
    # --- Multi-head self-attention (single head for clarity) ---
    Q = X @ W_q  # (seq_len, d_k)
    K = X @ W_k
    V = X @ W_v
    attn_out, attn_weights = scaled_dot_product_attention(Q, K, V)
    attn_out = attn_out @ W_o  # Project back to d_model

    # Residual + LayerNorm (Post-LN style: original paper)
    X = layer_norm(X + attn_out)

    # --- Feed-forward network: 2-layer MLP with ReLU ---
    ffn = np.maximum(0, X @ W_1 + b_1) @ W_2 + b_2  # (seq_len, d_model)

    # Residual + LayerNorm
    output = layer_norm(X + ffn)
    return output, attn_weights


# Initialize transformer block (d_model=32, d_k=32, d_ff=64)
d_model, d_k, d_ff = 32, 32, 64
seq_len = 10
rng = np.random.default_rng(0)
scale = 0.02

W_q = rng.standard_normal((d_model, d_k)) * scale
W_k = rng.standard_normal((d_model, d_k)) * scale
W_v = rng.standard_normal((d_model, d_k)) * scale
W_o = rng.standard_normal((d_k, d_model)) * scale
W_1 = rng.standard_normal((d_model, d_ff)) * scale
b_1 = np.zeros(d_ff)
W_2 = rng.standard_normal((d_ff, d_model)) * scale
b_2 = np.zeros(d_model)

X_input = rng.standard_normal((seq_len, d_model))
output, attn_weights = transformer_block(X_input, W_q, W_k, W_v, W_o, W_1, b_1, W_2, b_2)

print(f"Input shape:         {X_input.shape}")
print(f"Output shape:        {output.shape}")
print(f"Attention weights:   {attn_weights.shape}  (seq x seq, each row sums to 1)")
print(f"Attn weight row sum: {attn_weights[0].sum():.4f}  (should be 1.0)")
print(f"Output norm per pos: {np.linalg.norm(output, axis=1).mean():.4f}")

## Level 2: PyTorch TransformerEncoderLayer with OOM Handling

In [ ]:
# Use PyTorch's built-in TransformerEncoderLayer for production use.
# Add positional encoding and a classification head.

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding from 'Attention is All You Need'."""

    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, seq_len, d_model)"""
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerClassifier(nn.Module):
    """Transformer encoder with CLS-token pooling for classification."""

    def __init__(
        self, vocab_size: int, d_model: int = 64, nhead: int = 4,
        n_layers: int = 3, dim_ff: int = 256, n_classes: int = 2,
        dropout: float = 0.1, max_len: int = 128,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True  # Pre-LN is more stable
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x: torch.Tensor, src_key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        """x: (batch, seq_len) token ids."""
        x = self.pos_encoding(self.embedding(x))
        x = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        return self.classifier(x[:, 0, :])  # Use first token (CLS-style)


# Synthetic data
VOCAB, MAX_LEN, N = 500, 64, 1200
tokens = torch.randint(1, VOCAB, (N, MAX_LEN), device=device)
pad_mask = (tokens == 0)  # (batch, seq_len)
y_tfm = torch.randint(0, 2, (N,), device=device)

tr_loader_tfm = DataLoader(
    TensorDataset(tokens[:1000], pad_mask[:1000], y_tfm[:1000]), batch_size=64, shuffle=True
)
vl_loader_tfm = DataLoader(
    TensorDataset(tokens[1000:], pad_mask[1000:], y_tfm[1000:]), batch_size=64
)

tfm_model = TransformerClassifier(vocab_size=VOCAB).to(device)
tfm_opt = torch.optim.AdamW(tfm_model.parameters(), lr=1e-4, weight_decay=0.01)
tfm_crit = nn.CrossEntropyLoss()

total_p = sum(p.numel() for p in tfm_model.parameters())
print(f"TransformerClassifier: {total_p:,} parameters")

for epoch in range(10):
    tfm_model.train()
    for xb, mb, yb in tr_loader_tfm:
        try:
            tfm_opt.zero_grad()
            tfm_crit(tfm_model(xb, mb), yb).backward()
            nn.utils.clip_grad_norm_(tfm_model.parameters(), 1.0)
            tfm_opt.step()
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print("OOM — reduce batch_size or seq_len or d_model")
                torch.cuda.empty_cache(); continue
            raise

tfm_model.eval()
correct = total = 0
with torch.no_grad():
    for xb, mb, yb in vl_loader_tfm:
        correct += (tfm_model(xb, mb).argmax(1) == yb).sum().item(); total += len(yb)
print(f"Transformer encoder val accuracy: {correct/total:.3f}")

## Real-World Example 1: HuggingFace AutoModel Fine-Tuning

In [ ]:
# Production pattern: use HuggingFace AutoModel + AdamW + linear warmup schedule.
# We simulate the architecture since downloading real models requires internet access.
# The pattern below is directly transferable to bert-base-uncased, distilbert, roberta, etc.

class HuggingFaceStyleClassifier(nn.Module):
    """Simulates BertForSequenceClassification architecture."""

    def __init__(self, d_model: int = 768, n_classes: int = 2, n_layers: int = 12):
        super().__init__()
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model, nhead=12, dim_feedforward=3072, batch_first=True, dropout=0.1
            ),
            num_layers=n_layers,
        )
        self.pooler = nn.Linear(d_model, d_model)  # BERT pooler on [CLS]
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """hidden_states: (batch, seq_len, d_model) — pre-computed embeddings."""
        out = self.encoder(hidden_states)
        cls = torch.tanh(self.pooler(out[:, 0, :]))  # BERT-style [CLS] pooling
        return self.classifier(cls)


# Simulate pre-tokenized input (in practice: AutoTokenizer -> input_ids -> embedding)
batch_size, seq_len, d_model = 16, 64, 128  # Smaller d_model for speed
hidden = torch.randn(batch_size * 50, seq_len, d_model, device=device)
labels = torch.randint(0, 2, (batch_size * 50,), device=device)

hf_model = HuggingFaceStyleClassifier(d_model=d_model, n_classes=2, n_layers=2).to(device)
hf_loader = DataLoader(TensorDataset(hidden[:640], labels[:640]), batch_size=batch_size, shuffle=True)
hf_val = DataLoader(TensorDataset(hidden[640:800], labels[640:800]), batch_size=batch_size)

# AdamW + linear warmup (standard fine-tuning recipe)
from torch.optim.lr_scheduler import LambdaLR

total_steps = len(hf_loader) * 5
warmup_steps = int(0.1 * total_steps)
hf_opt = torch.optim.AdamW(hf_model.parameters(), lr=2e-5, weight_decay=0.01)

def warmup_linear_schedule(step: int) -> float:
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    return max(0.0, float(total_steps - step) / float(max(1, total_steps - warmup_steps)))

scheduler = LambdaLR(hf_opt, lr_lambda=warmup_linear_schedule)
hf_crit = nn.CrossEntropyLoss()

step = 0
for _ in range(5):
    hf_model.train()
    for xb, yb in hf_loader:
        hf_opt.zero_grad()
        hf_crit(hf_model(xb), yb).backward()
        nn.utils.clip_grad_norm_(hf_model.parameters(), 1.0)
        hf_opt.step(); scheduler.step(); step += 1

hf_model.eval()
correct = total = 0
with torch.no_grad():
    for xb, yb in hf_val:
        correct += (hf_model(xb).argmax(1) == yb).sum().item(); total += len(yb)
print(f"HuggingFace-style fine-tune val acc: {correct/total:.3f}")
print(f"Final LR: {hf_opt.param_groups[0]['lr']:.2e}")

## Real-World Example 2: Positional Encoding Ablation

In [ ]:
# Positional encoding gives the transformer knowledge of token order.
# Without it, the model is permutation-invariant (bag-of-words).
# Compare: sinusoidal PE, learned PE, and no PE on an order-sensitive task.

# Task: classify whether a sequence is in ascending order (order matters)
def generate_order_task(n: int, seq_len: int = 16, seed: int = 42) -> tuple:
    """Generate sequences. Label 1 if the sequence is 'mostly ascending'."""
    rng = np.random.default_rng(seed)
    seqs = rng.integers(1, 50, (n, seq_len))
    # Positive: sorted ascending; Negative: reversed or random
    labels = np.zeros(n, dtype=int)
    for i in range(n):
        if rng.random() < 0.5:
            seqs[i] = np.sort(seqs[i])  # Ascending
            labels[i] = 1
        else:
            seqs[i] = np.sort(seqs[i])[::-1]  # Descending
            labels[i] = 0
    return torch.tensor(seqs, dtype=torch.long), torch.tensor(labels, dtype=torch.long)


SEQ_LEN, VOCAB_PE, D = 16, 51, 32
X_pe, y_pe = generate_order_task(1200, SEQ_LEN)
X_pe, y_pe = X_pe.to(device), y_pe.to(device)
pe_tr = DataLoader(TensorDataset(X_pe[:1000], y_pe[:1000]), batch_size=64, shuffle=True)
pe_vl = DataLoader(TensorDataset(X_pe[1000:], y_pe[1000:]), batch_size=64)


def build_pe_model(use_pe: str = 'sinusoidal') -> nn.Module:
    """Build transformer with different positional encoding strategies."""
    embed = nn.Embedding(VOCAB_PE, D)
    if use_pe == 'sinusoidal':
        pos_enc = PositionalEncoding(D, SEQ_LEN, 0.0)
    elif use_pe == 'learned':
        pos_enc = nn.Embedding(SEQ_LEN, D)  # Learned absolute PE
    else:
        pos_enc = None

    class PEClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed = embed
            self.pe = pos_enc
            self.pe_type = use_pe
            self.encoder = nn.TransformerEncoder(
                nn.TransformerEncoderLayer(D, nhead=4, dim_feedforward=64, batch_first=True),
                num_layers=2
            )
            self.clf = nn.Linear(D, 2)

        def forward(self, x):
            e = self.embed(x)
            if self.pe_type == 'sinusoidal':
                e = self.pe(e)
            elif self.pe_type == 'learned':
                pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
                e = e + self.pe(pos)
            return self.clf(self.encoder(e).mean(1))
    return PEClassifier().to(device)


print(f"{'PE type':>12} | {'Val Acc':>9}")
print("-" * 28)
for pe_type in ['none', 'sinusoidal', 'learned']:
    m = build_pe_model(pe_type)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for _ in range(15):
        m.train()
        for xb, yb in pe_tr:
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
    m.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in pe_vl:
            correct += (m(xb).argmax(1) == yb).sum().item(); total += len(yb)
    print(f"{pe_type:>12} | {correct/total:9.3f}")

## Real-World Example 3: Transformer vs LSTM on Long Sequences

In [ ]:
# Transformer: O(T^2) attention — memory grows quadratically with sequence length.
# LSTM: O(T) memory — scales better for very long sequences.
# But transformer parallelizes across T while LSTM is sequential.

import time

VOCAB_COMP = 100
D_COMP = 64
BATCH_COMP = 16

class LSTMBaseline(nn.Module):
    def __init__(self, vocab: int, d: int = 64, n_classes: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.lstm = nn.LSTM(d, d, num_layers=2, batch_first=True, dropout=0.1)
        self.clf = nn.Linear(d, n_classes)

    def forward(self, x):
        _, (h_n, _) = self.lstm(self.embed(x))
        return self.clf(h_n[-1])


class TransformerBaseline(nn.Module):
    def __init__(self, vocab: int, d: int = 64, n_classes: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=128, batch_first=True),
            num_layers=2
        )
        self.clf = nn.Linear(d, n_classes)

    def forward(self, x):
        return self.clf(self.encoder(self.embed(x)).mean(1))


print(f"{'Seq Len':>9} | {'LSTM ms':>9} | {'TFM ms':>8} | {'LSTM OOM':>9} | {'TFM OOM':>8}")
print("-" * 55)
for seq_len in [32, 128, 256, 512, 1024]:
    x_dummy = torch.randint(1, VOCAB_COMP, (BATCH_COMP, seq_len), device=device)

    lstm_m = LSTMBaseline(VOCAB_COMP, D_COMP).to(device).eval()
    tfm_m = TransformerBaseline(VOCAB_COMP, D_COMP).to(device).eval()

    lstm_oom = tfm_oom = False
    lat_lstm = lat_tfm = float('nan')

    try:
        with torch.no_grad():
            lstm_m(x_dummy)  # warmup
            t0 = time.perf_counter()
            for _ in range(30): lstm_m(x_dummy)
            lat_lstm = (time.perf_counter() - t0) / 30 * 1000
    except RuntimeError as e:
        if "out of memory" in str(e).lower(): lstm_oom = True

    try:
        with torch.no_grad():
            tfm_m(x_dummy)  # warmup
            t0 = time.perf_counter()
            for _ in range(30): tfm_m(x_dummy)
            lat_tfm = (time.perf_counter() - t0) / 30 * 1000
    except RuntimeError as e:
        if "out of memory" in str(e).lower(): tfm_oom = True

    print(f"{seq_len:>9} | {lat_lstm:9.2f} | {lat_tfm:8.2f} | "
          f"{'OOM' if lstm_oom else 'OK':>9} | {'OOM' if tfm_oom else 'OK':>8}")

print("\nTransformer parallelizes over seq_len on GPU, LSTM must iterate step-by-step.")